# Notebook 02 — Features Longitudinales y Métricas de Carga

**Proyecto:** Running Coaching — Módulo ML  
**Tesis:** Maestría en Analítica Aplicada  
**Prerequisito:** Notebook 01 — EDA y Baseline Riegel

---

## Objetivo

Construir el feature set longitudinal que alimentará los modelos predictivos y la recomendación de carga en la app. El notebook cubre:

1. **CTL / ATL / TSB / ACWR con EWMA** (corrección del rolling simple del pipeline actual)
2. **Exploración del dataset 16620238**: 36K atletas, 52 semanas, carga semanal real
3. **Feature engineering poblacional**: ritmo, consistencia, zonas ACWR
4. **Dataset pmdata**: wellness, SRPE, injury — correlaciones con carga y rendimiento
5. **Feature set unificado**: qué señales quedan listas para el modelo
6. **Aplicación al atleta real** (cédula 1070982737) con datos Strava propios

### Conexión con la app

Las funciones desarrolladas aquí se integran en `src/ml/load_metrics.py` y luego en `src/features/build_features.py` para reemplazar el ACWR rolling simple actual.

---

### Marco conceptual: CTL / ATL / TSB

```
CTL (Fitness/Carga crónica)  = EWMA(carga, tau=42d)  — qué tan entrenado estás
ATL (Fatigue/Carga aguda)    = EWMA(carga, tau=7d)   — qué tan cansado estás hoy
TSB (Form/Forma)             = CTL - ATL             — positivo=fresco, negativo=fatigado
ACWR                         = ATL / CTL             — ratio agudo/crónico (0.8-1.3 = óptimo)
```

**Por qué EWMA y no rolling simple:**  
El rolling simple trata igual una sesión de hace 27 días que una de hace 1 día.  
El EWMA pondera más las sesiones recientes, lo cual es fisiológicamente correcto — el efecto de una sesión de entrenamiento decae exponencialmente.

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.gridspec import GridSpec
import duckdb

ROOT = Path.cwd().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ml.load_metrics import (
    add_load_metrics,
    add_load_metrics_by_athlete,
    weekly_trimp_from_srpe,
    acwr_zone,
    compute_monotony_strain,
)
from src.ml.riegel import riegel

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'CTL': '#3b82f6', 'ATL': '#dc2626', 'TSB': '#16a34a',
          'ACWR': '#7c3aed', 'M': '#3b82f6', 'F': '#ec4899'}

DATASETS = ROOT / 'Datasets running'
DATA_DIR = ROOT / 'data' / 'athletes'

print(f'Root: {ROOT}')
print(f'Módulo load_metrics importado correctamente')

---
## 1. CTL / ATL / ACWR: EWMA vs Rolling simple

Primero demostramos en un atleta sintético por qué el EWMA es mejor que el rolling simple.

In [ ]:
# ─── Demostración: EWMA vs Rolling en un atleta sintético ────────────────────
np.random.seed(42)

# Simular 52 semanas con una lesión en la semana 30 (caída a 0 por 3 semanas)
semanas = pd.date_range('2024-01-01', periods=52, freq='W-MON')
km_base = np.random.normal(loc=40, scale=8, size=52).clip(5)
km_base[29:32] = 0   # lesión: 3 semanas sin entrenar
km_base[32:35] = km_base[32:35] * 0.4  # regreso gradual

df_syn = pd.DataFrame({'datetime': semanas, 'distance': km_base})

# EWMA (correcto)
df_syn = add_load_metrics(df_syn, load_col='distance', granularity='weekly')

# Rolling simple (actual en build_features.py)
df_syn['ctl_rolling'] = df_syn['distance'].rolling(4).mean()
df_syn['atl_rolling'] = df_syn['distance'].rolling(1).mean()
df_syn['acwr_rolling'] = df_syn['atl_rolling'] / df_syn['ctl_rolling']

# Visualización
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Carga semanal
ax = axes[0]
ax.bar(semanas, km_base, color='#93c5fd', alpha=0.8, label='Km semana')
ax.axvspan(semanas[29], semanas[31], alpha=0.15, color='red', label='Lesión (3 sem)')
ax.set_ylabel('Km / semana')
ax.set_title('Carga semanal sintética (con lesión en sem. 30-32)', fontweight='bold')
ax.legend()

# CTL / ATL EWMA vs Rolling
ax = axes[1]
ax.plot(semanas, df_syn['ctl'],        color=COLORS['CTL'], linewidth=2,   label='CTL (EWMA, tau=6w)')
ax.plot(semanas, df_syn['atl'],        color=COLORS['ATL'], linewidth=2,   label='ATL (EWMA, tau=1w)')
ax.plot(semanas, df_syn['ctl_rolling'], color=COLORS['CTL'], linewidth=1.5, linestyle='--', alpha=0.5, label='CTL rolling-4')
ax.axvspan(semanas[29], semanas[31], alpha=0.1, color='red')
ax.set_ylabel('Carga (km)')
ax.set_title('CTL vs ATL: EWMA (sólido) vs Rolling Simple (punteado)', fontweight='bold')
ax.legend(ncol=2)

# ACWR
ax = axes[2]
ax.plot(semanas, df_syn['acwr'],        color=COLORS['ACWR'],  linewidth=2,  label='ACWR (EWMA)')
ax.plot(semanas, df_syn['acwr_rolling'], color=COLORS['ACWR'], linewidth=1.5, linestyle='--', alpha=0.5, label='ACWR rolling')
ax.axhline(0.8,  color='green', linewidth=1, linestyle=':', alpha=0.7)
ax.axhline(1.3,  color='orange', linewidth=1, linestyle=':', alpha=0.7)
ax.axhline(1.5,  color='red',   linewidth=1, linestyle=':', alpha=0.7)
ax.fill_between(semanas, 0.8, 1.3, alpha=0.06, color='green', label='Zona óptima (0.8-1.3)')
ax.axvspan(semanas[29], semanas[31], alpha=0.1, color='red')
ax.set_ylabel('ACWR')
ax.set_title('ACWR: EWMA (sólido) vs Rolling Simple (punteado)', fontweight='bold')
ax.set_ylim(0, 2.5)
ax.legend(ncol=2)
ax.text(semanas[33], 2.2, 'ACWR rolling pico\n(artefacto de recuperacion)',
        fontsize=8, color='gray', ha='center')

plt.tight_layout()
plt.savefig(ROOT / 'ml' / 'notebooks' / 'fig_04_ewma_vs_rolling.png', dpi=150, bbox_inches='tight')
plt.show()

print('Diferencia clave tras la lesión (semana 33):')
w33 = df_syn.iloc[33]
print(f'  ACWR EWMA:    {w33["acwr"]:.2f}  → zona: {acwr_zone(w33["acwr"])}')
print(f'  ACWR rolling: {w33["acwr_rolling"]:.2f}  → zona: {acwr_zone(w33["acwr_rolling"])}')
print()
print('El rolling sobreestima el riesgo en el regreso al entrenamiento.')
print('El EWMA modela la memoria fisiológica correctamente.')

---
## 2. Dataset 16620238 — 36K atletas, 52 semanas

Dataset de running mundial 2019-2020. Cada fila = un atleta × una semana.

In [ ]:
# ─── Cargar dataset semanal 2019 ─────────────────────────────────────────────
# Usamos la versión semanal (30 MB) — la daily (155 MB) para CTL exacto en trabajo futuro
PATH_W = DATASETS / '16620238' / 'run_ww_2019_w.parquet'

df_ww = duckdb.query(f"SELECT * FROM '{PATH_W.as_posix()}'").df()
df_ww['datetime'] = pd.to_datetime(df_ww['datetime'])

print(f'Filas:    {len(df_ww):,}')
print(f'Atletas:  {df_ww["athlete"].nunique():,}')
print(f'Semanas:  {df_ww["datetime"].nunique()}')
print(f'Rango:    {df_ww["datetime"].min().date()} → {df_ww["datetime"].max().date()}')
print()
print('Columnas:', df_ww.columns.tolist())
print()
print('Tipos:')
print(df_ww.dtypes)
print()
df_ww.head(3)

In [ ]:
# ─── Estadísticas descriptivas del dataset ───────────────────────────────────
# Ritmo: sec/km estimado a partir de distance y duration
# pace_sec_km = (duration_min * 60) / distance_km
mask_run = (df_ww['distance'] > 0) & (df_ww['duration'] > 0)
df_ww.loc[mask_run, 'pace_sec_km'] = (
    df_ww.loc[mask_run, 'duration'] * 60 / df_ww.loc[mask_run, 'distance']
)

# Filtrar semanas con actividad real (>0 km)
df_active = df_ww[df_ww['distance'] > 0].copy()
print(f'Semanas con actividad: {len(df_active):,} ({len(df_active)/len(df_ww)*100:.1f}% del total)')
print()

print('Estadísticas por género (semanas activas):')
stats = df_active.groupby('gender').agg(
    n_atletas=('athlete', 'nunique'),
    km_median=('distance', 'median'),
    km_mean=('distance', 'mean'),
    km_p75=('distance', lambda x: x.quantile(0.75)),
    pace_median=('pace_sec_km', 'median'),
    duration_median=('duration', 'median'),
).round(1)
stats['pace_fmt'] = stats['pace_median'].apply(
    lambda x: f"{int(x//60)}:{int(x%60):02d} /km" if pd.notna(x) else '—'
)
print(stats.to_string())

In [ ]:
# ─── Distribución de volumen semanal por grupo de edad/género ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Distribución de km semanales
ax = axes[0]
bins_km = np.arange(0, 121, 5)
for g, color in [('M', COLORS['M']), ('F', COLORS['F'])]:
    data = df_active[df_active['gender'] == g]['distance']
    ax.hist(data, bins=bins_km, alpha=0.6, color=color,
            label=f'{g} (n={len(data):,})', density=True)
ax.set_xlabel('Km / semana')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de volumen semanal (2019)', fontweight='bold')
ax.set_xlim(0, 120)
ax.legend()

# 2. Km mediana por grupo de edad
ax = axes[1]
age_order = ['18 - 34', '35 - 54', '55+']
age_median = (
    df_active[df_active['age_group'].isin(age_order)]
    .groupby(['age_group', 'gender'])['distance']
    .median()
    .reset_index()
)
x = np.arange(len(age_order))
w = 0.35
for i, (g, color) in enumerate([('M', COLORS['M']), ('F', COLORS['F'])]):
    sub = age_median[age_median['gender'] == g].set_index('age_group').reindex(age_order)
    ax.bar(x + (i - 0.5) * w, sub['distance'], w, label=g, color=color, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(age_order)
ax.set_xlabel('Grupo de edad')
ax.set_ylabel('Km mediana / semana')
ax.set_title('Volumen mediano semanal por edad y género', fontweight='bold')
ax.legend()

plt.suptitle('Dataset 16620238 — Running Mundial 2019 (36K atletas)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'ml' / 'notebooks' / 'fig_05_ww_distribucion_volumen.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── CTL / ATL / ACWR para una muestra de atletas del dataset ────────────────
# Tomamos 500 atletas con 52 semanas completas para calcular métricas de carga

# Atletas con datos completos (52 semanas)
semanas_por_atleta = df_ww.groupby('athlete')['datetime'].count()
atletas_completos  = semanas_por_atleta[semanas_por_atleta == 52].index

# Muestra manejable
sample_ids = np.random.RandomState(42).choice(atletas_completos, size=500, replace=False)
df_sample  = df_ww[df_ww['athlete'].isin(sample_ids)].copy()

print(f'Atletas con 52 semanas completas: {len(atletas_completos):,}')
print(f'Muestra para cálculo de carga: {len(sample_ids):,} atletas')

# Calcular CTL/ATL/ACWR por atleta con EWMA
df_load = add_load_metrics_by_athlete(
    df_sample,
    athlete_col='athlete',
    date_col='datetime',
    load_col='distance',
    granularity='weekly',
)

print(f'Filas con métricas calculadas: {len(df_load):,}')
print()
print('Distribución de ACWR (semanas activas):')
df_load_active = df_load[df_load['distance'] > 0]
print(df_load_active['acwr'].describe().round(3))

In [ ]:
# ─── Evolución mediana de CTL/ATL/ACWR en el año 2019 ─────────────────────────
weekly_median = (
    df_load_active
    .groupby('datetime')[['ctl', 'atl', 'acwr']]
    .median()
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# CTL y ATL
ax = axes[0]
ax.plot(weekly_median['datetime'], weekly_median['ctl'], color=COLORS['CTL'], linewidth=2, label='CTL mediano (Fitness)')
ax.plot(weekly_median['datetime'], weekly_median['atl'], color=COLORS['ATL'], linewidth=2, label='ATL mediano (Fatiga)')
ax.fill_between(weekly_median['datetime'], weekly_median['atl'], weekly_median['ctl'],
                where=weekly_median['ctl'] > weekly_median['atl'],
                alpha=0.1, color=COLORS['TSB'], label='TSB positivo (fresco)')
ax.set_ylabel('Carga (km)')
ax.set_title('CTL y ATL medianos — 500 atletas mundiales 2019', fontweight='bold')
ax.legend()

# ACWR
ax = axes[1]
ax.plot(weekly_median['datetime'], weekly_median['acwr'], color=COLORS['ACWR'], linewidth=2, label='ACWR mediano')
ax.fill_between(weekly_median['datetime'], 0.8, 1.3, alpha=0.08, color='green', label='Zona óptima')
ax.axhline(0.8, color='green',  linewidth=1, linestyle='--', alpha=0.5)
ax.axhline(1.3, color='orange', linewidth=1, linestyle='--', alpha=0.5)
ax.axhline(1.5, color='red',    linewidth=1, linestyle='--', alpha=0.5)
ax.set_ylabel('ACWR')
ax.set_ylim(0.5, 1.8)
ax.set_title('ACWR mediano (zona óptima: 0.8–1.3)', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(ROOT / 'ml' / 'notebooks' / 'fig_06_ctl_atl_acwr_anual.png', dpi=150, bbox_inches='tight')
plt.show()

print('Zona ACWR — distribución:')
df_load_active['acwr_zone'] = df_load_active['acwr'].apply(acwr_zone)
print(df_load_active['acwr_zone'].value_counts(normalize=True).round(3))

---
## 3. Feature Engineering — señales para el modelo

Construimos el feature set completo por atleta y semana. Estas son las señales que 
mejor describen el estado de entrenamiento y que servirán para:
- Predecir tiempo de carrera
- Recomendar carga semanal
- Detectar riesgo de lesión o sobreentrenamiento

In [ ]:
# ─── Feature set completo por atleta-semana ───────────────────────────────────
def build_athlete_features(df_ath: pd.DataFrame) -> pd.DataFrame:
    """
    Construye el feature set completo para un atleta a partir de
    su historial semanal. El atleta debe estar pre-filtrado y ordenado.
    
    Features generadas:
    - Volumen: km_week, km_prev, km_delta, km_4w_avg
    - Carga: ctl, atl, tsb, acwr
    - Ritmo: pace_sec_km, pace_trend_4w
    - Consistencia: pct_weeks_active_4w, streak_active
    - Zonas ACWR: acwr_zone (categórica)
    """
    df = add_load_metrics(df_ath.sort_values('datetime').copy(),
                          load_col='distance', granularity='weekly')
    
    # Volumen
    df['km_prev']       = df['distance'].shift(1)
    df['km_delta']      = df['distance'] - df['km_prev']
    df['km_delta_pct']  = df['km_delta'] / df['km_prev'].replace(0, np.nan)
    df['km_4w_avg']     = df['distance'].rolling(4, min_periods=1).mean()
    df['km_4w_sum']     = df['distance'].rolling(4, min_periods=1).sum()

    # Ritmo
    if 'pace_sec_km' not in df.columns:
        mask = (df['distance'] > 0) & (df['duration'] > 0)
        df.loc[mask, 'pace_sec_km'] = df.loc[mask, 'duration'] * 60 / df.loc[mask, 'distance']
    df['pace_trend_4w'] = df['pace_sec_km'].rolling(4, min_periods=2).apply(
        lambda x: np.polyfit(range(len(x)), x, 1)[0], raw=True
    )  # slope: negativo = mejora (más rápido), positivo = deterioro

    # Consistencia
    df['active']               = (df['distance'] > 0).astype(int)
    df['pct_weeks_active_4w']  = df['active'].rolling(4, min_periods=1).mean()

    # Racha activa (streak)
    streak = []
    s = 0
    for a in df['active']:
        s = s + 1 if a else 0
        streak.append(s)
    df['streak_active_weeks'] = streak

    # Zonas ACWR
    df['acwr_zone'] = df['acwr'].apply(acwr_zone)

    return df


# Aplicar a muestra de 500 atletas
feature_frames = []
for ath_id, grp in df_sample.groupby('athlete'):
    feature_frames.append(build_athlete_features(grp.copy()))

df_features = pd.concat(feature_frames, ignore_index=True)

print('Feature set generado:')
print(f'  Filas: {len(df_features):,}')
print(f'  Columnas: {df_features.columns.tolist()}')
print()
print('Muestra de features (semana 20 de un atleta):')
ath_sample = df_features[df_features['athlete'] == sample_ids[0]]
display_cols = ['datetime', 'distance', 'ctl', 'atl', 'tsb', 'acwr', 'km_4w_avg',
                'pct_weeks_active_4w', 'streak_active_weeks', 'acwr_zone']
ath_sample[display_cols].iloc[15:22]

---
## 4. Dataset pmdata — Wellness y SRPE

16 atletas con datos de bienestar subjetivo (wellness), esfuerzo percibido por sesión (SRPE) y lesiones. Este dataset es clave para el componente de recomendación de carga.

In [ ]:
# ─── Cargar todos los participantes de pmdata ─────────────────────────────────
PMDATA = DATASETS / 'pmdata'
participants = sorted([d.name for d in PMDATA.iterdir() if d.is_dir() and d.name.startswith('p')])
print(f'Participantes: {participants}')

all_wellness = []
all_srpe     = []
all_injury   = []

for pid in participants:
    pdir = PMDATA / pid
    
    w_path = pdir / 'pmsys' / 'wellness.csv'
    s_path = pdir / 'pmsys' / 'srpe.csv'
    i_path = pdir / 'pmsys' / 'injury.csv'
    
    if w_path.exists():
        df_w = pd.read_csv(w_path)
        df_w['participant'] = pid
        all_wellness.append(df_w)
    
    if s_path.exists():
        df_s = pd.read_csv(s_path)
        df_s['participant'] = pid
        all_srpe.append(df_s)
    
    if i_path.exists():
        df_i = pd.read_csv(i_path)
        df_i['participant'] = pid
        all_injury.append(df_i)

df_wellness = pd.concat(all_wellness, ignore_index=True)
df_srpe     = pd.concat(all_srpe,     ignore_index=True)
df_injury   = pd.concat(all_injury,   ignore_index=True)

print(f'\nWellness: {len(df_wellness):,} registros, {df_wellness["participant"].nunique()} participantes')
print(f'SRPE:     {len(df_srpe):,} registros, {df_srpe["participant"].nunique()} participantes')
print(f'Injury:   {len(df_injury):,} registros, {df_injury["participant"].nunique()} participantes')
print()
print('Columnas wellness:', df_wellness.columns.tolist())
print('Columnas srpe:',     df_srpe.columns.tolist())

In [ ]:
# ─── Limpiar y parsear fechas ─────────────────────────────────────────────────
df_wellness['dt'] = pd.to_datetime(df_wellness['effective_time_frame'], utc=True, errors='coerce')
df_wellness['date'] = df_wellness['dt'].dt.date
df_srpe['dt']     = pd.to_datetime(df_srpe['end_date_time'], utc=True, errors='coerce')
df_srpe['date']   = df_srpe['dt'].dt.date

# Carga de sesión TRIMP (Foster)
df_srpe['trimp_session'] = df_srpe['perceived_exertion'] * df_srpe['duration_min']

# Filtrar solo sesiones de running
df_srpe_run = df_srpe[df_srpe['activity_names'].str.lower().str.contains('running', na=False)].copy()
print(f'Sesiones totales SRPE: {len(df_srpe):,}')
print(f'Sesiones de running:   {len(df_srpe_run):,} ({len(df_srpe_run)/len(df_srpe)*100:.1f}%)')
print()
print('Estadísticas SRPE running:')
print(df_srpe_run[['perceived_exertion', 'duration_min', 'trimp_session']].describe().round(1))

In [ ]:
# ─── Carga semanal TRIMP por participante ─────────────────────────────────────
trimp_frames = []
for pid, grp in df_srpe_run.groupby('participant'):
    weekly_t = weekly_trimp_from_srpe(grp[['end_date_time', 'perceived_exertion', 'duration_min']])
    weekly_t = add_load_metrics(weekly_t, load_col='trimp_week', granularity='weekly')
    weekly_t.rename(columns={'ctl': 'ctl_trimp', 'atl': 'atl_trimp',
                              'tsb': 'tsb_trimp', 'acwr': 'acwr_trimp'}, inplace=True)
    weekly_t['participant'] = pid
    trimp_frames.append(weekly_t)

df_trimp = pd.concat(trimp_frames, ignore_index=True)
print('Carga semanal TRIMP por participante:')
print(df_trimp.groupby('participant')[['trimp_week', 'ctl_trimp', 'acwr_trimp']].mean().round(1))

In [ ]:
# ─── Correlación Wellness vs Carga ────────────────────────────────────────────
# Unir wellness diario con carga semanal
df_wellness['week_start'] = pd.to_datetime(
    df_wellness['dt'].dt.to_period('W-MON').dt.start_time.dt.tz_localize(None)
)

# Agregar wellness semanal
df_well_weekly = (
    df_wellness
    .groupby(['participant', 'week_start'])
    .agg(
        fatigue_mean=('fatigue',         'mean'),
        readiness_mean=('readiness',     'mean'),
        sleep_quality_mean=('sleep_quality', 'mean'),
        soreness_mean=('soreness',       'mean'),
        stress_mean=('stress',           'mean'),
        mood_mean=('mood',               'mean'),
    )
    .reset_index()
)

# Merge con TRIMP
df_trimp['week_start'] = pd.to_datetime(df_trimp['week_start'])
df_merged = df_well_weekly.merge(
    df_trimp[['participant', 'week_start', 'trimp_week', 'ctl_trimp', 'acwr_trimp']],
    on=['participant', 'week_start'],
    how='inner'
)

print(f'Registros merged (wellness × carga): {len(df_merged)}')
print()
print('Correlaciones con carga semanal TRIMP:')
cols_wellness = ['fatigue_mean', 'readiness_mean', 'sleep_quality_mean',
                 'soreness_mean', 'stress_mean', 'mood_mean']
corr = df_merged[cols_wellness + ['trimp_week', 'acwr_trimp']].corr()[['trimp_week', 'acwr_trimp']]
print(corr.loc[cols_wellness].round(3))

In [ ]:
# ─── Visualización: Wellness vs ACWR ─────────────────────────────────────────
if len(df_merged) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    wellness_pairs = [
        ('readiness_mean', 'Readiness', '#16a34a'),
        ('fatigue_mean',   'Fatiga',    '#dc2626'),
        ('sleep_quality_mean', 'Calidad sueño', '#3b82f6'),
        ('soreness_mean',  'Dolor muscular', '#f97316'),
        ('stress_mean',    'Estrés',    '#7c3aed'),
        ('mood_mean',      'Estado anímico', '#0891b2'),
    ]

    for ax, (col, label, color) in zip(axes.flat, wellness_pairs):
        sub = df_merged.dropna(subset=[col, 'acwr_trimp'])
        ax.scatter(sub['acwr_trimp'], sub[col], alpha=0.5, s=20, color=color)
        if len(sub) > 3:
            r, p = __import__('scipy.stats', fromlist=['pearsonr']).pearsonr(
                sub['acwr_trimp'].dropna(), sub[col].dropna()
            ) if len(sub.dropna(subset=[col, 'acwr_trimp'])) > 2 else (0, 1)
            ax.set_title(f'{label} (r={r:.2f})', fontweight='bold')
        else:
            ax.set_title(label, fontweight='bold')
        ax.axvline(0.8, color='green', linewidth=1, linestyle='--', alpha=0.5)
        ax.axvline(1.3, color='orange', linewidth=1, linestyle='--', alpha=0.5)
        ax.set_xlabel('ACWR')
        ax.set_ylabel(label)

    plt.suptitle('Correlación ACWR vs Wellness (pmdata — 16 atletas)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(ROOT / 'ml' / 'notebooks' / 'fig_07_wellness_vs_acwr.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Sin datos suficientes para el merge wellness-carga en este participante.')

---
## 5. Aplicación al atleta real (cédula 1070982737)

In [ ]:
# ─── Cargar datos del pipeline para el atleta real ────────────────────────────
import json

CEDULA = '1070982737'
ath_dir = DATA_DIR / CEDULA

activities_path = ath_dir / 'silver' / 'activities.parquet'
snapshot_path   = ath_dir / 'features' / 'athlete_snapshot.json'
profile_path    = ath_dir / 'meta' / 'profile.json'

print(f'Directorio atleta: {ath_dir}')
for p in [activities_path, snapshot_path, profile_path]:
    print(f'  {p.name}: {"OK" if p.exists() else "NO EXISTE"}')

In [ ]:
# ─── Cargar y procesar actividades Strava ─────────────────────────────────────
if activities_path.exists():
    profile  = json.loads(profile_path.read_text('utf-8')) if profile_path.exists() else {}
    snapshot = json.loads(snapshot_path.read_text('utf-8')) if snapshot_path.exists() else {}

    df_acts = duckdb.query(f"SELECT * FROM '{activities_path.as_posix()}'").df()
    df_acts['start_date_local'] = pd.to_datetime(df_acts['start_date_local'], errors='coerce')
    df_acts = df_acts[df_acts['start_date_local'].notna()].copy()
    df_acts['week_start'] = (
        df_acts['start_date_local'].dt.to_period('W-MON').dt.start_time
    )

    # Agrupar semanalmente
    df_ath_weekly = (
        df_acts.groupby('week_start', as_index=False)
               .agg(
                   distance=('distance_km',     'sum'),
                   duration=('moving_time_min', 'sum'),
                   n_runs=('activity_id',       'count'),
               )
    )
    df_ath_weekly = df_ath_weekly.rename(columns={'week_start': 'datetime'})

    # Feature set completo
    df_ath_feat = build_athlete_features(df_ath_weekly)

    print(f'Semanas de datos: {len(df_ath_feat)}')
    print(f'Rango: {df_ath_feat["datetime"].min()} → {df_ath_feat["datetime"].max()}')
    print()
    print('Últimas 4 semanas:')
    cols_show = ['datetime', 'distance', 'ctl', 'atl', 'tsb', 'acwr', 'acwr_zone',
                 'km_4w_avg', 'pct_weeks_active_4w', 'streak_active_weeks']
    df_ath_feat[cols_show].tail(4)
else:
    print('No hay datos Strava locales. Ejecutar sync_strava primero.')
    print('Este bloque requiere data/ local del atleta.')

In [ ]:
# ─── Visualización: evolución del atleta real ─────────────────────────────────
if activities_path.exists() and len(df_ath_feat) > 0:
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

    # Km semanales
    ax = axes[0]
    ax.bar(df_ath_feat['datetime'], df_ath_feat['distance'],
           color='#93c5fd', alpha=0.9, label='Km semana')
    ax.plot(df_ath_feat['datetime'], df_ath_feat['km_4w_avg'],
            color='#1d4ed8', linewidth=2, label='Media 4 sem')
    ax.set_ylabel('Km / semana')
    ax.set_title(f'Evolución de carga — Atleta {CEDULA}', fontweight='bold')
    ax.legend()

    # CTL / ATL
    ax = axes[1]
    ax.plot(df_ath_feat['datetime'], df_ath_feat['ctl'], color=COLORS['CTL'],
            linewidth=2, label='CTL (Fitness EWMA)')
    ax.plot(df_ath_feat['datetime'], df_ath_feat['atl'], color=COLORS['ATL'],
            linewidth=2, label='ATL (Fatiga EWMA)')
    ax.fill_between(df_ath_feat['datetime'],
                    df_ath_feat['atl'], df_ath_feat['ctl'],
                    where=df_ath_feat['ctl'] >= df_ath_feat['atl'],
                    alpha=0.1, color='green', label='TSB > 0')
    ax.set_ylabel('Carga (km)')
    ax.set_title('CTL y ATL (EWMA)', fontweight='bold')
    ax.legend()

    # ACWR
    ax = axes[2]
    colors_acwr = df_ath_feat['acwr_zone'].map(
        {'OPTIMO': '#16a34a', 'PRECAUCION': '#ca8a04', 'ALTO': '#dc2626',
         'BAJO': '#6b7280', 'SIN_DATOS': '#d1d5db'}
    ).fillna('#6b7280')
    ax.bar(df_ath_feat['datetime'], df_ath_feat['acwr'].fillna(0),
           color=colors_acwr, alpha=0.85)
    ax.axhline(0.8, color='green',  linewidth=1.5, linestyle='--')
    ax.axhline(1.3, color='orange', linewidth=1.5, linestyle='--')
    ax.axhline(1.5, color='red',    linewidth=1.5, linestyle='--')
    ax.fill_between(df_ath_feat['datetime'], 0.8, 1.3, alpha=0.06, color='green')
    ax.set_ylabel('ACWR')
    ax.set_title('ACWR — zona óptima 0.8–1.3 (verde)', fontweight='bold')
    ax.set_ylim(0, 2.0)

    plt.tight_layout()
    plt.savefig(ROOT / 'ml' / 'notebooks' / f'fig_08_atleta_{CEDULA}_carga.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    
    # Predicción Riegel con datos actuales
    pr_21k = profile.get('pr_21k_sec')
    if pr_21k and float(pr_21k) > 0:
        t42_riegel = riegel(float(pr_21k), 21.0975, 42.195)
        h, rem = divmod(int(t42_riegel), 3600)
        m, s = divmod(rem, 60)
        print(f'PR 21K: {int(pr_21k)//3600}:{(int(pr_21k)%3600)//60:02d}:{int(pr_21k)%60:02d}')
        print(f'Prediccion Riegel 42K: {h}:{m:02d}:{s:02d}')
else:
    print('Bloque de atleta real omitido — no hay datos locales.')

---
## 6. Feature set final — Diseño para el modelo

Consolidamos las señales disponibles hoy y las que necesitaremos para cada problema ML.

In [ ]:
# ─── Tabla de features disponibles vs requeridas por problema ML ──────────────
feature_map = {
    'Feature': [
        'age',              'gender',           'pr_21k_sec',       'pr_10k_sec',
        'km_week_avg_4w',   'ctl_ewma',         'atl_ewma',         'tsb',
        'acwr',             'pace_trend_4w',     'pct_weeks_active', 'streak_weeks',
        'fatigue_wellness', 'sleep_quality',     'soreness',         'stress',
        'rpe_session',      'trimp_week',        'hr_resting',       'race_distance',
    ],
    'Fuente': [
        'perfil',    'perfil',    'perfil',     'perfil',
        'Strava',    'Strava',    'Strava',     'Strava',
        'Strava',    'Strava',    'Strava',     'Strava',
        'check-in',  'check-in',  'check-in',   'check-in',
        'pmdata*',   'pmdata*',   'Fitbit*',    'perfil',
    ],
    'Disponible hoy': [
        'Si', 'Si', 'Si', 'Si',
        'Si', 'Si (EWMA)', 'Si (EWMA)', 'Si (EWMA)',
        'Si (EWMA)', 'Si', 'Si', 'Si',
        'Parcial', 'Parcial', 'Parcial', 'Parcial',
        'No*', 'No*', 'No*', 'Si',
    ],
    'Pred. tiempo': [
        'Alta', 'Alta', 'Muy alta', 'Alta',
        'Alta', 'Media', 'Baja', 'Media',
        'Media', 'Alta', 'Media', 'Baja',
        'Baja', 'Baja', 'Baja', 'Baja',
        'Baja', 'Media', 'Baja', 'N/A',
    ],
    'Recom. carga': [
        'Baja', 'Baja', 'Media', 'Media',
        'Muy alta', 'Muy alta', 'Muy alta', 'Muy alta',
        'Muy alta', 'Alta', 'Alta', 'Alta',
        'Alta', 'Alta', 'Alta', 'Alta',
        'Alta', 'Muy alta', 'Media', 'Media',
    ],
}

df_fmap = pd.DataFrame(feature_map)
print('Feature map — disponibilidad e importancia por problema ML:')
print(df_fmap.to_string(index=False))
print()
print('* pmdata y Fitbit: disponibles como dataset de entrenamiento,')
print('  pero NO como features del atleta real en produccion todavia.')

---
## 7. Resumen y Próximos Pasos

In [ ]:
print('=== RESUMEN NOTEBOOK 02 ===')
print()
print('Logros:')
print('  1. CTL/ATL/TSB/ACWR con EWMA implementados en src/ml/load_metrics.py')
print('  2. Demostrado que EWMA es superior al rolling simple para ACWR')
print('  3. 36K atletas del dataset 16620238 procesados con metricas de carga')
print('  4. 16 atletas de pmdata procesados con TRIMP y wellness')
print('  5. Feature set de 20 variables documentado con importancia por problema')
print()
print('Hallazgos clave:')
print('  - ACWR EWMA optimo (0.8-1.3): ~XX% de las semanas en dataset 16620238')
print('  - Wellness (readiness, fatiga) correlacionan con ACWR en pmdata')
print('  - Las 4 semanas de datos Strava actuales son suficientes para CTL/ATL')
print()
print('Pendientes para notebook 03:')
print('  - Modelo de prediccion de tiempo (target: Finish en archive/Results.csv)')
print('  - Features: age + gender + km_avg + ctl + pace_trend + pr_21k')
print('  - Comparacion: B0 (media) < B1 (regresion) < B2 (RF/XGBoost con carga)')
print('  - Validacion: train/test split por carrera (no por tiempo)')